# 04 — Layerwise analysis across all teacher–student pairs

Notebook này tái sử dụng hoặc huấn luyện GATE-KD và TALAS trên cả ba cặp mô hình, sau đó tạo hai hình CKA layerwise từ cùng một probe held-out và cùng ba seed.

Hai đầu ra gồm raw linear CKA và thay đổi CKA so với pretrained student. Các heatmap là phân tích mô tả; chúng không được diễn giải như bằng chứng nhân quả về cơ chế tối ưu.

Caption trong paper cần nói rõ các trained panels là trung bình của ba seed còn pretrained student là một checkpoint xác định; $\Delta$CKA lấy pretrained student làm mốc; embedding-output layer 0 bị loại vì centered CKA không xác định tại representation hằng; mọi panel cùng loại dùng chung color scale.

In [1]:
# 1. Cấu hình thí nghiệm
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

REPO_URL = "https://github.com/duncan-nguyen/embedding-kd.git"
AUTO_PULL_REPO = True
INSTALL_REQUIREMENTS = True

PAIR_ORDER = [
    "qwen3_4b_to_bert_base",
    "bge_m3_to_minilm_h768",
    "qwen3_0.6b_to_minilm_h384",
]
PAIR_LABELS = {
    "qwen3_4b_to_bert_base": "Qwen3-4B → BERT-base",
    "bge_m3_to_minilm_h768": "BGE-M3 → MiniLM-768",
    "qwen3_0.6b_to_minilm_h384": "Qwen3-0.6B → MiniLM-384",
}
TRAIN_DATA_REL = Path("data/train_set/merged_3_data_5k_each.csv")
VALIDATION_FILES = [
    Path("data/val_set/banking77_validation.csv"),
    Path("data/val_set/tweet_validation.csv"),
    Path("data/val_set/emotion_validation.csv"),
    Path("data/val_set/mrpc_validation.csv"),
    Path("data/val_set/scitail_validation.csv"),
    Path("data/val_set/wic_validation.csv"),
    Path("data/val_set/sick_validation.csv"),
    Path("data/val_set/sts12_validation.csv"),
    Path("data/val_set/stsb_validation.csv"),
]

SEEDS = [42, 43, 44]
BATCH_SIZE = 128
EPOCHS = 5
MAX_LENGTH = 256
NUM_WORKERS = 2
LRS = {"ours": 7e-5, "talas": 2e-5}
LAMBDA_TOPO = 0.5
TOPO_CLOUD_SIZE = 128

PROBE_SIZE = 512
PROBE_SEED = 0
ENCODE_BATCH_SIZE = 64
DEVICE = "cuda"
ANALYSIS_DEVICE = "cuda"

EXECUTE = True
STOP_ON_ERROR = True
REQUIRE_ALL_RUNS = True
RENDER_FIGURES = True
COPY_TO_PAPER = False
SAVE_TO_GOOGLE_DRIVE = False
CUDA_VISIBLE_DEVICES = "0"
EXPECTED_GPU = "H200"
MAX_PARALLEL_JOBS = 3
GPUS = None

RUN_STAMP = datetime.now(ZoneInfo("Asia/Ho_Chi_Minh")).strftime("%Y%m%d-%H%M%S")
# Điền tên run cũ để resume sau khi runtime restart; None tạo run mới.
RUN_NAME_OVERRIDE = "analysis_layerwise_all_pairs_talas15k_v1"
RUN_NAME = RUN_NAME_OVERRIDE or f"analysis_layerwise_all_pairs_{RUN_STAMP}"

assert SEEDS and len(SEEDS) == len(set(SEEDS))
assert TOPO_CLOUD_SIZE == BATCH_SIZE
print(f"Pairs: {len(PAIR_ORDER)}; seeds: {SEEDS}")
print(f"Run: {RUN_NAME}")

Pairs: 3; seeds: [42, 43, 44]
Run: analysis_layerwise_all_pairs_talas15k_v1


In [2]:
# 2. Dùng repo hiện tại hoặc clone trên Colab; cài dependencies
import subprocess
import sys

cwd = Path.cwd().resolve()
PROJECT_DIR = next(
    (path for path in (cwd, cwd.parent) if (path / "main.py").is_file()),
    None,
)
if PROJECT_DIR is None:
    clone_parent = Path("/content") if Path("/content").is_dir() else cwd
    PROJECT_DIR = clone_parent / "embedding-kd"
    if not PROJECT_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
    assert (PROJECT_DIR / "main.py").is_file(), f"Repo không hợp lệ: {PROJECT_DIR}"

tracked = subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "status", "--porcelain", "--untracked-files=no"],
    check=True, capture_output=True, text=True,
).stdout.strip()
if AUTO_PULL_REPO and not tracked:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)
elif AUTO_PULL_REPO:
    print("[git] Bỏ qua pull vì repo có tracked changes.")

if INSTALL_REQUIREMENTS:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt")],
        check=True,
    )

git_head = subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "rev-parse", "--short", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()
sys.path[:0] = [str(PROJECT_DIR), str(PROJECT_DIR / "notebooks")]
print(f"Repo: {PROJECT_DIR} @ {git_head}")

Repo: /content/embedding-kd @ c0c446e


In [3]:
# 3. Output, GPU, dữ liệu và probe held-out
import os
import pandas as pd
import torch
from IPython.display import display
from _layerwise_analysis import build_heldout_probe

try:
    from google.colab import drive as colab_drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True

if IN_COLAB and SAVE_TO_GOOGLE_DRIVE:
    colab_drive.mount("/content/drive")
    OUTPUT_BASE = Path("/content/drive/MyDrive/embedding-kd-runs")
else:
    OUTPUT_BASE = PROJECT_DIR / "runs"

RUN_ROOT = OUTPUT_BASE / RUN_NAME
CACHE_DIR = OUTPUT_BASE / "teacher_cache"
TRAIN_DATA = PROJECT_DIR / TRAIN_DATA_REL
PROBE_PATH = RUN_ROOT / "probe_texts.csv"
RUN_ROOT.mkdir(parents=True, exist_ok=True)

assert TRAIN_DATA.is_file(), f"Thiếu training data: {TRAIN_DATA}"
for relative in VALIDATION_FILES:
    assert (PROJECT_DIR / relative).is_file(), f"Thiếu validation data: {relative}"
if EXECUTE or RENDER_FIGURES:
    assert torch.cuda.is_available(), "Hãy bật GPU runtime trước khi chạy."
    gpu_names = [torch.cuda.get_device_properties(i).name for i in range(torch.cuda.device_count())]
    if hasattr(torch.cuda, "is_bf16_supported"):
        assert torch.cuda.is_bf16_supported(), "GPU phải hỗ trợ BF16 để khớp main runs."
    for index in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(index)
        print(f"cuda:{index}: {props.name} ({props.total_memory / 2**30:.1f} GiB)")

if PROBE_PATH.is_file():
    probe_table = pd.read_csv(PROBE_PATH)
    assert len(probe_table) == PROBE_SIZE, (
        f"Probe cache có {len(probe_table)} hàng, nhưng cấu hình yêu cầu {PROBE_SIZE}."
    )
    probe_texts = probe_table["text"].astype(str).tolist()
    print(f"[probe] reuse {PROBE_PATH}")
else:
    probe_texts, probe_table = build_heldout_probe(
        PROJECT_DIR, TRAIN_DATA, VALIDATION_FILES,
        size=PROBE_SIZE, seed=PROBE_SEED,
    )
    probe_table.to_csv(PROBE_PATH, index=False)
    print(f"[probe] wrote {PROBE_PATH}")

display(probe_table.groupby("source").size().rename("sentences").to_frame())
print(f"Training data: {TRAIN_DATA}")
print(f"Output root: {RUN_ROOT}")

cuda:0: NVIDIA RTX PRO 6000 Blackwell Server Edition (95.0 GiB)
[probe] wrote /content/embedding-kd/runs/analysis_layerwise_all_pairs_talas15k_v1/probe_texts.csv


,sentences
source,
banking77_validation,41
emotion_validation,64
mrpc_validation,24
scitail_validation,62
sick_validation,33
sts12_validation,40
stsb_validation,110
tweet_validation,96
wic_validation,42


Training data: /content/embedding-kd/data/train_set/merged_3_data_5k_each.csv
Output root: /content/embedding-kd/runs/analysis_layerwise_all_pairs_talas15k_v1


In [4]:
# 4. Tạo plan — hai method, ba seed và ba cặp mô hình
import shlex
from _analysis_common import PAIRS, distill_command

unknown_pairs = sorted(set(PAIR_ORDER) - set(PAIRS))
assert not unknown_pairs, f"Pair chưa được khai báo: {unknown_pairs}"
JOBS = []
for pair_key in PAIR_ORDER:
    pair = PAIRS[pair_key]
    for arm, method in (("ours", "geoode"), ("talas", "talas")):
        for seed in SEEDS:
            run_dir = RUN_ROOT / "runs" / pair_key / arm / f"seed_{seed}"
            extra = ["--no_eval_retrieval"]
            if arm == "ours":
                extra.extend([
                    "--lambda_topo", str(LAMBDA_TOPO),
                    "--lambda_ctr", "0.0",
                    "--gauge_refit_every", "1",
                    "--topo_batch_size", str(TOPO_CLOUD_SIZE),
                ])
            command = distill_command(
                PROJECT_DIR,
                method=method,
                pair=pair,
                train_data=TRAIN_DATA,
                cache_dir=CACHE_DIR,
                run_dir=run_dir,
                seed=seed,
                batch_size=BATCH_SIZE,
                epochs=EPOCHS,
                learning_rate=LRS[arm],
                max_length=MAX_LENGTH,
                num_workers=NUM_WORKERS,
                extra=extra,
            )
            JOBS.append({
                "name": f"{pair_key}/{arm}/seed_{seed}",
                "pair": pair_key,
                "arm": arm,
                "seed": seed,
                "run_dir": run_dir,
                "command": command,
            })

print(f"Plan: {len(PAIR_ORDER)} pairs × 2 methods × {len(SEEDS)} seeds = {len(JOBS)} jobs")
for job in JOBS:
    print(f"[{job['name']}] {shlex.join(job['command'])}")

Plan: 3 pairs × 2 methods × 3 seeds = 18 jobs
[qwen3_4b_to_bert_base/ours/seed_42] /usr/bin/python3 /content/embedding-kd/main.py --method geoode --train_data /content/embedding-kd/data/train_set/merged_3_data_5k_each.csv --student_model google-bert/bert-base-uncased --teacher_model Qwen/Qwen3-Embedding-4B --teacher_pooling last_token --batch_size 128 --epochs 5 --save_every 5 --lr 7e-05 --max_length 256 --seed 42 --num_workers 2 --eval_every 0 --pair_threshold_source test --cache_dir /content/embedding-kd/runs/teacher_cache --save_dir /content/embedding-kd/runs/analysis_layerwise_all_pairs_talas15k_v1/runs/qwen3_4b_to_bert_base/ours/seed_42 --no_wandb --student_pooling cls --no_eval_retrieval --lambda_topo 0.5 --lambda_ctr 0.0 --gauge_refit_every 1 --topo_batch_size 128
[qwen3_4b_to_bert_base/ours/seed_43] /usr/bin/python3 /content/embedding-kd/main.py --method geoode --train_data /content/embedding-kd/data/train_set/merged_3_data_5k_each.csv --student_model google-bert/bert-base-unca

In [ ]:
# 5. Chạy hoặc resume các jobs; final-test record là resume boundary
from _analysis_common import collect_jobs, prewarm_teacher_cache, run_jobs

if EXECUTE:
    # Warm từng teacher cache trước khi fan-out để tránh nhiều process cùng encode teacher.
    if MAX_PARALLEL_JOBS > 1 or (GPUS is not None and len(GPUS) > 1):
        for pair_key in PAIR_ORDER:
            prewarm_teacher_cache(
                PROJECT_DIR,
                pair=PAIRS[pair_key],
                train_data=TRAIN_DATA,
                cache_dir=CACHE_DIR,
                max_length=MAX_LENGTH,
                cuda_visible_devices=CUDA_VISIBLE_DEVICES,
            )
    run_status = run_jobs(
        PROJECT_DIR,
        JOBS,
        cuda_visible_devices=CUDA_VISIBLE_DEVICES,
        stop_on_error=STOP_ON_ERROR,
        max_parallel=MAX_PARALLEL_JOBS,
        gpus=GPUS,
    )
    display(run_status)
else:
    print("Dry run: đặt EXECUTE=True để chạy các checkpoint còn thiếu.")

run_audit = collect_jobs(JOBS)
display(run_audit[["pair", "arm", "seed", "status", "run_dir"]])
missing_runs = run_audit.loc[run_audit["status"] != "done"]
if REQUIRE_ALL_RUNS and not missing_runs.empty:
    raise RuntimeError(f"Còn {len(missing_runs)} run chưa hoàn tất; chưa tạo heatmap.")

[cache] building /content/embedding-kd/runs/teacher_cache/qwen-qwen3-embedding-4b__merged_3_data_5k_each__last_token__49d848173270.pt
[cache] building /content/embedding-kd/runs/teacher_cache/baai-bge-m3__merged_3_data_5k_each__cls__23ad45061e9d.pt
[cache] building /content/embedding-kd/runs/teacher_cache/qwen-qwen3-embedding-0-6b__merged_3_data_5k_each__last_token__48d210faccc5.pt
[parallel] 18 job(s) over 3 slot(s): ['0', '0', '0']
[START 1/18] qwen3_4b_to_bert_base/ours/seed_42 on GPU 0 (pid 7635) -> /content/embedding-kd/runs/analysis_layerwise_all_pairs_talas15k_v1/runs/qwen3_4b_to_bert_base/ours/seed_42/train.log
[START 2/18] qwen3_4b_to_bert_base/ours/seed_43 on GPU 0 (pid 7636) -> /content/embedding-kd/runs/analysis_layerwise_all_pairs_talas15k_v1/runs/qwen3_4b_to_bert_base/ours/seed_43/train.log
[START 3/18] qwen3_4b_to_bert_base/ours/seed_44 on GPU 0 (pid 7637) -> /content/embedding-kd/runs/analysis_layerwise_all_pairs_talas15k_v1/runs/qwen3_4b_to_bert_base/ours/seed_44/train

In [ ]:
# 6. Trích hidden states và cache riêng cho từng checkpoint
from _layerwise_analysis import encode_pair_grid

ENCODING_ROOT = RUN_ROOT / "encodings"
ENCODED = {}
for pair_key in PAIR_ORDER:
    print(f"\n[encode] {PAIR_LABELS[pair_key]}")
    ENCODED[pair_key] = encode_pair_grid(
        pair_key=pair_key,
        pair=PAIRS[pair_key],
        pair_run_root=RUN_ROOT / "runs" / pair_key,
        seeds=SEEDS,
        epochs=EPOCHS,
        texts=probe_texts,
        encoding_root=ENCODING_ROOT,
        device=DEVICE,
        batch_size=ENCODE_BATCH_SIZE,
        max_length=MAX_LENGTH,
    )
print(f"Encoded layers: {ENCODING_ROOT}")

In [ ]:
# 7. Tính raw CKA và $\Delta$CKA so với pretrained student
from _layerwise_analysis import analyze_pair

ANALYSIS_ROOT = RUN_ROOT / "analysis"
RESULTS = {}
summary_rows = []
for pair_key in PAIR_ORDER:
    print(f"\n[analysis] {PAIR_LABELS[pair_key]}")
    result = analyze_pair(
        pair_key=pair_key,
        encoded_paths=ENCODED[pair_key],
        seeds=SEEDS,
        analysis_root=ANALYSIS_ROOT,
        device=ANALYSIS_DEVICE,
    )
    RESULTS[pair_key] = result
    for arm in ("ours", "talas"):
        summary_rows.append({
            "pair": pair_key,
            "arm": arm,
            "final-final CKA": result[arm]["cka_mean"][-1, -1],
            "final-final ΔCKA": result[arm]["delta_cka"][-1, -1],
        })

analysis_summary = pd.DataFrame(summary_rows)
analysis_summary.to_csv(RUN_ROOT / "layerwise_summary.csv", index=False)
display(analysis_summary.round(4))

In [ ]:
# 8. Render hai hình CKA cho cả ba cặp
import shutil
from IPython.display import Image
from _layerwise_analysis import render_cka_figures

FIGURE_DIR = RUN_ROOT / "figures"
figure_files = []
if RENDER_FIGURES:
    figure_files = render_cka_figures(
        RESULTS,
        pair_order=PAIR_ORDER,
        pair_labels=PAIR_LABELS,
        output_dir=FIGURE_DIR,
    )
    for path in figure_files:
        print(path)
    for path in sorted(FIGURE_DIR.glob("*.png")):
        display(Image(filename=str(path)))

    if COPY_TO_PAPER:
        paper_dir = PROJECT_DIR / "docs" / "latex_iclr" / "figures" / "layerwise_all_pairs"
        paper_dir.mkdir(parents=True, exist_ok=True)
        for path in figure_files:
            shutil.copy2(path, paper_dir / path.name)
        print(f"Copied figures to {paper_dir}")

In [ ]:
# 9. Ghi manifest để audit và tái lập kết quả
import hashlib
import json

probe_sha256 = hashlib.sha256(PROBE_PATH.read_bytes()).hexdigest()
manifest = {
    "git_head": git_head,
    "run_name": RUN_NAME,
    "gpu_names": gpu_names if (EXECUTE or RENDER_FIGURES) else [],
    "cuda_visible_devices": CUDA_VISIBLE_DEVICES,
    "pairs": PAIR_ORDER,
    "seeds": SEEDS,
    "training": {
        "data": str(TRAIN_DATA_REL),
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rates": LRS,
        "lambda_topo": LAMBDA_TOPO,
        "topo_cloud_size": TOPO_CLOUD_SIZE,
        "max_length": MAX_LENGTH,
    },
    "probe": {
        "path": str(PROBE_PATH),
        "sha256": probe_sha256,
        "size": PROBE_SIZE,
        "seed": PROBE_SEED,
        "validation_files": [str(path) for path in VALIDATION_FILES],
    },
    "analysis": {
        "metrics": ["linear_cka", "delta_linear_cka"],
        "omitted_embedding_output_layer": 0,
    },
    "figures": [str(path) for path in figure_files],
    "runs": run_audit.to_dict(orient="records"),
}
manifest_path = RUN_ROOT / "experiment_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")
run_audit.to_csv(RUN_ROOT / "run_audit.csv", index=False)
print(f"Manifest: {manifest_path}")
print(f"Matrices: {ANALYSIS_ROOT}")
print(f"Figures: {FIGURE_DIR}")